In [4]:
!pip install -q -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 28.8 MB/s eta 0:00:00


In [5]:
pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 3.4 MB/s eta 0:00:00a 0:00:01
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=2884b1e664b1e7aff430f17bf7250919c896e5d5bc524be6068af1d5226a41a6
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
Note: you may need to restart the kernel to use updated packages.


In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [7]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor_smol= AutoProcessor.from_pretrained(MODEL_NAME)

model_smol = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_smol.eval()

print("SmolVLM2 loaded successfully!")

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.64k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

SmolVLM2 loaded successfully!


In [8]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "/images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_name"] == target_name:
            return record["query"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [9]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [10]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [11]:

def evaluate(query_text, b64_str):
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_smol.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_smol(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
    ).to(model_smol.device)

    with torch.inference_mode():
        output_ids = model_smol.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_smol.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()



In [12]:
import transformers
print(transformers.__version__)

5.19.0


In [13]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/images/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Create a lookup map for ground truth: (image_id) -> label
# Since we are evaluating the actual files in /images/, we target '/images/' records
ground_truth_map = {}
for record in annotations_list:
    if record["image_name"].startswith("/images/"):
        img_id = record["image_id"]
        ground_truth_map[img_id] = record["label"]

# Get list of all images in the directory
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
        
        
    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val})

df_results = pd.DataFrame(results)
display(df_results.head(10))

Processing 500 images through all variations...


  0%|          | 0/500 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

,Image_Id,Ground_truth,Original
0,169361,No,No.
1,47022,No,Yes.
2,556064,No,No.
3,474004,No,Yes.
4,306710,No,Yes.
5,33882,No,No.
6,195213,No,Yes.
7,196545,No,No.
8,186703,No,No.
9,215863,No,No.


In [16]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/evaluations/Image_degradation_evalautions.csv")

merged_df = pd.merge(df, df_results, on="Image_Id", how="outer")
merged_df.head()


,Image_Id,Ground_truth_x,Original_x,Standard_Blur,Heavy_Blur,Occlusion,JPEG_Compression,Resizing,Ground_truth_y,Original_y
0,6578,No,Yes.,Yes.,No,Yes.,Yes.,No,No,Yes.
1,8602,No,No,No,No,No,Yes.,No,No,No.
2,9469,No,No,No,No,No,No,No,No,No.
3,9712,No,Yes.,No,Yes.,Yes,No,No,No,No.
4,10733,No,No,No,No,No,No,No.,No,No.


In [19]:
merged_df.rename(columns={
    "Ground_truth_x" : "Ground_truth",
    "Original_x": "Qwen-3B",
    "Original_y": "Smol-2.2B"
}, inplace=True)

merged_df = merged_df[["Image_Id", "Ground_truth" ,"Qwen-3B", "Smol-2.2B"]]
merged_df.to_csv("diff_models_eval.csv", index=False)

/tmp/ipykernel_49/4285335250.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df.rename(columns={


In [20]:
merged_df.head()

,Image_Id,Ground_truth,Qwen-3B,Smol-2.2B
0,6578,No,Yes.,Yes.
1,8602,No,No,No.
2,9469,No,No,No.
3,9712,No,Yes.,No.
4,10733,No,No,No.
